In [ ]:
from google.colab import drive
drive.mount('/content/drive')

NLP_PATH = '/content/drive/MyDrive/Colab Notebooks/NLP-Costumer-Classifier/'
print("Drive connected!")
print(f"Path: {NLP_PATH}")

In [ ]:
!pip install streamlit plotly -q
print("✅ Libraries installed!")

In [ ]:
import os

print("Checking required files in Drive...")
files = ['tfidf.pkl', 'lr_model.pkl', 'nlp_data.pkl']

all_good = True
for f in files:
    path = NLP_PATH + f
    if os.path.exists(path):
        size = os.path.getsize(path) / (1024*1024)
        print(f"  ✅ {f} ({size:.1f} MB)")
    else:
        print(f"  ❌ {f} NOT FOUND")
        all_good = False

if all_good:
    print("\n✅ All files found!")
else:
    print("\n❌ Missing files — run Notebook 2 and 3 first")

In [ ]:
import shutil
import os

print("Copying files to /content/...")

files_to_copy = ['tfidf.pkl', 'lr_model.pkl', 'nlp_data.pkl']

for fname in files_to_copy:
    src  = NLP_PATH + fname
    dest = '/content/' + fname
    if os.path.exists(src):
        shutil.copy(src, dest)
        print(f"  ✅ Copied: {fname}")
    else:
        print(f"  ❌ Missing in Drive: {fname}")

print("\nAll files ready in /content/")

In [ ]:
# Write app.py as separate lines — no escape issues
lines = [
"import streamlit as st",
"import pandas as pd",
"import plotly.express as px",
"import joblib",
"import pickle",
"import re",
"import nltk",
"from nltk.corpus import stopwords",
"from nltk.stem import WordNetLemmatizer",
"",
"nltk.download('stopwords', quiet=True)",
"nltk.download('wordnet', quiet=True)",
"",
"tfidf    = joblib.load('tfidf.pkl')",
"lr_model = joblib.load('lr_model.pkl')",
"",
"with open('nlp_data.pkl', 'rb') as f:",
"    data = pickle.load(f)",
"",
"le = data['le']",
"df = data['df']",
"",
"st.set_page_config(page_title='Complaint Classifier', layout='wide')",
"st.title('Banking Complaint Classifier')",
"st.markdown('Powered by TF-IDF and DistilBERT')",
"st.markdown('---')",
"",
"st.sidebar.title('Navigation')",
"page = st.sidebar.radio('Go to', ['Classify Complaint', 'Analytics', 'About'])",
"",
"if page == 'Classify Complaint':",
"    st.header('Classify a Customer Complaint')",
"    complaint = st.text_area('Enter complaint:', height=180)",
"    if st.button('Classify'):",
"        if complaint.strip():",
"            stop_words = set(stopwords.words('english'))",
"            lemmatizer = WordNetLemmatizer()",
"            text = complaint.lower()",
"            text = re.sub('[^a-zA-Z ]', '', text)",
"            words = [lemmatizer.lemmatize(w) for w in text.split() if w not in stop_words]",
"            clean = ' '.join(words)",
"            vec       = tfidf.transform([clean])",
"            pred      = lr_model.predict(vec)[0]",
"            proba     = lr_model.predict_proba(vec)[0]",
"            category  = le.inverse_transform([pred])[0]",
"            conf      = proba.max() * 100",
"            col1, col2 = st.columns(2)",
"            col1.metric('Category',   category)",
"            col2.metric('Confidence', f'{conf:.1f}%')",
"            proba_df = pd.DataFrame({'Category': le.classes_, 'Score': proba * 100})",
"            proba_df = proba_df.sort_values('Score', ascending=True)",
"            fig = px.bar(proba_df, x='Score', y='Category', orientation='h',",
"                         title='Confidence by Category', color='Score',",
"                         color_continuous_scale='reds')",
"            st.plotly_chart(fig, use_container_width=True)",
"        else:",
"            st.warning('Please enter a complaint first.')",
"",
"elif page == 'Analytics':",
"    st.header('Complaint Analytics')",
"    col1, col2, col3 = st.columns(3)",
"    col1.metric('Total Complaints', f'{len(df):,}')",
"    col2.metric('Categories', df['category'].nunique())",
"    col3.metric('Avg Length', f\"{int(df['text'].str.len().mean())} chars\")",
"    st.markdown('---')",
"    cat_counts = df['category'].value_counts().reset_index()",
"    cat_counts.columns = ['category', 'count']",
"    col1, col2 = st.columns(2)",
"    with col1:",
"        fig1 = px.bar(cat_counts, x='category', y='count',",
"                      title='Complaints by Category',",
"                      color='count', color_continuous_scale='reds')",
"        fig1.update_xaxes(tickangle=45)",
"        st.plotly_chart(fig1, use_container_width=True)",
"    with col2:",
"        fig2 = px.pie(cat_counts, names='category', values='count',",
"                      title='Category Share', hole=0.4)",
"        st.plotly_chart(fig2, use_container_width=True)",
"    df['text_length'] = df['text'].str.len()",
"    fig3 = px.histogram(df, x='text_length', nbins=50,",
"                        title='Complaint Length Distribution',",
"                        color_discrete_sequence=['#e74c3c'])",
"    fig3.update_xaxes(range=[0, 5000])",
"    st.plotly_chart(fig3, use_container_width=True)",
"",
"elif page == 'About':",
"    st.header('About This Project')",
"    st.markdown('---')",
"    st.markdown('### NLP Customer Complaint Classifier')",
"    st.markdown('End-to-end NLP pipeline on 160K+ CFPB banking complaints.')",
"    st.markdown('---')",
"    st.markdown('### Models')",
"    st.markdown('- TF-IDF + Logistic Regression (baseline)')",
"    st.markdown('- DistilBERT (advanced)')",
"    st.markdown('- DistilBERT SST-2 (sentiment)')",
"    st.markdown('---')",
"    st.markdown('### Tech Stack')",
"    st.markdown('Python | HuggingFace | Scikit-learn | Streamlit | Plotly | NLTK')",
]

with open('/content/app.py', 'w') as f:
    f.write('\n'.join(lines))

with open(NLP_PATH + 'app.py', 'w') as f:
    f.write('\n'.join(lines))

print("✅ app.py written cleanly!")

# Verify it loads
result = __import__('subprocess').run(
    ['python', '-c', 'import ast; ast.parse(open("app.py").read()); print("Syntax OK!")'],
    capture_output=True, text=True, cwd='/content'
)
print(result.stdout)
print(result.stderr if result.stderr else "No errors!")

In [ ]:
import subprocess
import time
from google.colab.output import eval_js

subprocess.run(['pkill', '-f', 'streamlit'], capture_output=True)
time.sleep(3)

proc = subprocess.Popen([
    'streamlit', 'run', '/content/app.py',
    '--server.port', '8501',
    '--server.headless', 'true'
])

time.sleep(10)

print("="*50)
print("✅ DASHBOARD IS LIVE!")
print("="*50)
print("\n🌐 Click this link:")
print(eval_js("google.colab.kernel.proxyPort(8501)"))

In [ ]:
# Run this to keep dashboard alive
# Stop cell to shut down

import time

print("Dashboard is running...")
print("Stop this cell to shut it down.")
print("-"*40)

for i in range(999):
    time.sleep(60)
    print(f"  Running... {i+1} min")

In [ ]:
from google.colab import files

download_list = ['app.py', 'tfidf.pkl', 'lr_model.pkl', 'nlp_data.pkl']

for fname in download_list:
    files.download('/content/' + fname)
    print(f"✅ Downloaded: {fname}")